# LCEL Deep Dive — LangChain Expression Language

Master the pipe operator and advanced composition patterns: parallel execution, routing, fallbacks, and custom logic.

---

In [ ]:
!pip install -q langchain langchain-openai langchain-anthropic

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
    RunnableBranch,
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1 · RunnablePassthrough — Forward Inputs Unchanged

Pass data through without modification. Essential for combining raw input with chain outputs.

In [ ]:
# Passthrough forwards input as-is
passthrough = RunnablePassthrough()
print(passthrough.invoke({"topic": "RAG"}))  # {"topic": "RAG"}

In [ ]:
# Real use case: carry original input alongside a chain's output
prompt = ChatPromptTemplate.from_template("Summarize this in 1 sentence: {topic}")
summary_chain = prompt | llm | StrOutputParser()

# .assign() adds new keys while keeping existing ones
chain_with_passthrough = RunnablePassthrough.assign(
    summary=summary_chain
)

result = chain_with_passthrough.invoke({"topic": "Transformer architecture"})
print(f"Original topic: {result['topic']}")
print(f"Summary:        {result['summary']}")

## 2 · RunnableParallel — Execute Chains Simultaneously

Run multiple chains on the same input in parallel. Each key becomes a field in the output dict.

In [ ]:
# Three analyses on the same input, executed in parallel
pros_prompt = ChatPromptTemplate.from_template("List 3 pros of {technology}. Be concise.")
cons_prompt = ChatPromptTemplate.from_template("List 3 cons of {technology}. Be concise.")
usecase_prompt = ChatPromptTemplate.from_template("Give 3 use cases for {technology}. Be concise.")

analysis = RunnableParallel(
    pros=pros_prompt | llm | StrOutputParser(),
    cons=cons_prompt | llm | StrOutputParser(),
    use_cases=usecase_prompt | llm | StrOutputParser(),
)

result = analysis.invoke({"technology": "LangChain"})

for key, value in result.items():
    print(f"\n=== {key.upper()} ===\n{value}")

## 3 · RunnableLambda — Inject Custom Python Logic

Wrap any Python function as a chain component. This is how you add preprocessing, validation, or transformations.

In [ ]:
# Custom preprocessing function
def clean_input(data: dict) -> dict:
    """Normalize and validate user input before hitting the LLM."""
    return {
        "query": data["query"].strip().lower(),
        "word_count": len(data["query"].split()),
    }

# Custom postprocessing function
def format_output(text: str) -> dict:
    """Structure the raw LLM output."""
    return {
        "answer": text,
        "char_count": len(text),
        "truncated": text[:100] + "..." if len(text) > 100 else text,
    }

prompt = ChatPromptTemplate.from_template("Answer concisely: {query}")

# Full pipeline: preprocess → prompt → LLM → parse → postprocess
chain = (
    RunnableLambda(clean_input)
    | prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(format_output)
)

result = chain.invoke({"query": "  What is RLHF?  "})
for k, v in result.items():
    print(f"{k}: {v}")

In [ ]:
# You can also use the @chain decorator for cleaner syntax
from langchain_core.runnables import chain as chain_decorator

@chain_decorator
def enrich_query(input_dict: dict) -> str:
    """Add context to a user query before sending to LLM."""
    query = input_dict["query"]
    context = f"The user is asking about AI/ML. Their question: {query}"
    response = llm.invoke(context)
    return response.content

print(enrich_query.invoke({"query": "What is LoRA?"}))

## 4 · RunnableBranch — Conditional Routing

Route inputs to different chains based on conditions. Like a switch statement for LLM pipelines.

In [ ]:
# Define specialized chains for different query types
code_prompt = ChatPromptTemplate.from_template(
    "You are a Python expert. Help with this coding question: {query}"
)
math_prompt = ChatPromptTemplate.from_template(
    "You are a math tutor. Solve this step by step: {query}"
)
general_prompt = ChatPromptTemplate.from_template(
    "Answer this question helpfully: {query}"
)

# Route based on input content
router = RunnableBranch(
    # (condition_function, chain_to_run)
    (lambda x: any(kw in x["query"].lower() for kw in ["code", "python", "function", "class"]),
     code_prompt | llm | StrOutputParser()),
    
    (lambda x: any(kw in x["query"].lower() for kw in ["calculate", "solve", "equation", "math"]),
     math_prompt | llm | StrOutputParser()),
    
    # Default fallback (no condition)
    general_prompt | llm | StrOutputParser(),
)

# Test routing
queries = [
    "Write a Python function to reverse a linked list",
    "Solve the equation 3x + 7 = 22",
    "What is the capital of France?",
]

for q in queries:
    result = router.invoke({"query": q})
    print(f"\nQ: {q}\nA: {result[:120]}...\n" + "-"*60)

## 5 · Fallbacks — Graceful Model Failures

If the primary model fails (rate limit, timeout, error), automatically fall back to an alternative.

In [ ]:
from langchain_anthropic import ChatAnthropic

# Primary: GPT-4o-mini | Fallback: Claude Sonnet
primary = ChatOpenAI(model="gpt-4o-mini", temperature=0)
fallback = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)

# .with_fallbacks() chains alternatives
resilient_llm = primary.with_fallbacks([fallback])

prompt = ChatPromptTemplate.from_template("Explain {concept} in 2 sentences.")
chain = prompt | resilient_llm | StrOutputParser()

# If GPT fails, Claude answers instead — zero downtime
print(chain.invoke({"concept": "attention mechanism"}))

In [ ]:
# Simulate a failure to see fallback in action
broken_llm = ChatOpenAI(model="gpt-nonexistent", temperature=0)  # will fail
safe_llm = broken_llm.with_fallbacks([fallback])

chain = prompt | safe_llm | StrOutputParser()

# This works because Claude catches the failure
print(chain.invoke({"concept": "RLHF"}))

## 6 · Putting It All Together — A Real Pipeline

Combine everything: preprocess → route → parallel analysis → postprocess, with fallbacks.

In [ ]:
# Preprocessing: clean and classify the input
def classify_and_clean(data: dict) -> dict:
    query = data["query"].strip()
    # Simple keyword-based classification
    if any(kw in query.lower() for kw in ["compare", "vs", "versus", "difference"]):
        category = "comparison"
    elif any(kw in query.lower() for kw in ["how to", "tutorial", "build", "create"]):
        category = "tutorial"
    else:
        category = "general"
    return {"query": query, "category": category}

# Different chains for different categories
comparison_chain = (
    ChatPromptTemplate.from_template(
        "Compare the following concisely in a structured way: {query}"
    ) | llm | StrOutputParser()
)

tutorial_chain = (
    ChatPromptTemplate.from_template(
        "Give a brief step-by-step guide: {query}"
    ) | llm | StrOutputParser()
)

general_chain = (
    ChatPromptTemplate.from_template(
        "Answer this concisely: {query}"
    ) | llm | StrOutputParser()
)

# Route based on classification
router = RunnableBranch(
    (lambda x: x["category"] == "comparison", comparison_chain),
    (lambda x: x["category"] == "tutorial", tutorial_chain),
    general_chain,  # default
)

# Full pipeline: clean → classify → route → answer
full_pipeline = RunnableLambda(classify_and_clean) | router

# Test with different query types
test_queries = [
    "Compare FAISS vs ChromaDB for vector search",
    "How to build a RAG pipeline",
    "What is an embedding model?",
]

for q in test_queries:
    result = full_pipeline.invoke({"query": q})
    print(f"\nQ: {q}\nA: {result[:150]}...\n" + "="*60)

---

## Key Takeaways

| Runnable | Purpose | When to Use |
|----------|---------|-------------|
| `RunnablePassthrough` | Forward input unchanged | Carry original data alongside chain output |
| `.assign()` | Add new keys to dict | Enrich input with computed fields |
| `RunnableParallel` | Execute chains simultaneously | Multiple analyses on same input |
| `RunnableLambda` | Inject custom Python | Pre/post processing, validation |
| `RunnableBranch` | Conditional routing | Different chains for different inputs |
| `.with_fallbacks()` | Graceful failure handling | Production resilience |

**Next →** [03 · Output Parsers](../03-output-parsers/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*